# Notebook 7: Complete System (RAG + TAG + Memory)## Features:- ✅ PDF document search (RAG)- ✅ Genre classification (TAG)- ✅ Web search- ✅ Conversational memory- ✅ All tools combined!

In [ ]:
%pip install langchain langchain-community langgraph python-dotenv openpyxl pandas langchain-openai google-search-results pypdf faiss-cpu --quiet

In [ ]:
import osfrom dotenv import load_dotenvfrom langchain_core.tools import toolfrom langchain_community.utilities import GoogleSerperAPIWrapperfrom langgraph.graph import StateGraph, END, MessagesStatefrom langgraph.prebuilt import ToolNodefrom langgraph.checkpoint.memory import MemorySaverfrom langchain_core.messages import HumanMessage, SystemMessagefrom ai_course_utils import (    load_llm_from_env,    load_use_case_config,    load_taxonomy_from_excel,    load_pdf_for_rag,    display_config)load_dotenv()load_dotenv('../.env')print("✓ Imports successful")

In [ ]:
display_config()

In [ ]:
use_case_file = "../data/use_case_Movie_Recommendation.xlsx"use_case_config = load_use_case_config(use_case_file)system_prompt = use_case_config.get("agent_prompt")

## Load All Data

In [ ]:
# Load taxonomytaxonomy_file = "../data/Movie_Recommendation_Taxonomy_File.xlsx"taxonomy = load_taxonomy_from_excel(taxonomy_file)# Load PDFpdf_file = "../data/The_98th_Academy_Awards___2026.pdf"documents, vectorstore, retriever = load_pdf_for_rag(pdf_file)print(f"\n✅ All data loaded!")print(f"  Genres: {len(taxonomy)}")print(f"  PDF Pages: {len(documents)}")

## Define All Tools

In [ ]:
@tooldef search_web(query: str) -> str:    """Search web for current information."""    search = GoogleSerperAPIWrapper()    return search.run(query)@tooldef search_documents(query: str) -> str:    """Search Oscars 2026 PDF."""    docs = retriever.invoke(query)    return "\n\n".join([doc.page_content for doc in docs])@tooldef classify_genre(query: str) -> dict:    """Classify query into movie genre."""    query_lower = query.lower()    for genre_name, info in taxonomy.items():        included = info.get('included', '').lower()        keywords = [k.strip() for k in included.split(',') if k.strip()]        if any(kw in query_lower for kw in keywords):            return {'genre': genre_name, 'guidelines': info.get('guidelines', '')}    return {'genre': 'General'}tools = [classify_genre, search_documents, search_web]print(f"✓ All 3 tools ready: {[t.name for t in tools]}")

In [ ]:
llm = load_llm_from_env()model = llm.bind_tools(tools)

## Build Complete Agent with Memory

In [ ]:
enhanced_prompt = """You are an expert movie assistant with three tools:1. classify_genre: Understand user preferences (10 custom genres)2. search_documents: Search 2026 Oscars nominations3. search_web: Find current movie informationAlways classify first, then search documents for Oscars, then web for general info."""def call_model(state: MessagesState):    return {"messages": [model.invoke([SystemMessage(content=enhanced_prompt)] + state["messages"])]}def should_continue(state: MessagesState):    return "tools" if state["messages"][-1].tool_calls else ENDworkflow = StateGraph(MessagesState)workflow.add_node("agent", call_model)workflow.add_node("tools", ToolNode(tools))workflow.set_entry_point("agent")workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})workflow.add_edge("tools", "agent")app = workflow.compile(checkpointer=MemorySaver())print("✓ Complete system with memory ready")

## Chat Helper Function

In [ ]:
def chat(user_input: str, thread_id: str = "default") -> str:    """Chat with the complete assistant (maintains memory)."""    config = {"configurable": {"thread_id": thread_id}}    result = None    for event in app.stream({"messages": [HumanMessage(content=user_input)]}, config, stream_mode="values"):        result = event    return result["messages"][-1].contentprint("🎬 Complete System (RAG + TAG + Memory) Ready!")

## Test Complete Conversations

In [ ]:
# Complex conversation using all capabilitiesthread = "complete"print("User: I love mythology mixed with sci-fi. Are there Oscar nominees like that?")r1 = chat("I love mythology mixed with sci-fi. Are there Oscar nominees like that?", thread_id=thread)print(f"Bot: {r1[:250]}...")print("\nUser: Tell me more about the first one you mentioned")r2 = chat("Tell me more about the first one you mentioned", thread_id=thread)print(f"Bot: {r2[:250]}...")print("\nUser: When are the Oscars?")r3 = chat("When are the Oscars?", thread_id=thread)print(f"Bot: {r3[:250]}...")